# 01 – Data Acquisition: Hourly Temperature Observations (2012)

## Objective

This notebook retrieves raw hourly temperature observations for Oklahoma (starting with Oklahoma City) for the year 2012.

We will collect raw observational data from:

- NOAA surface observation stations
- Oklahoma Mesonet stations
- Potential additional observational networks

The goal of this notebook is **not analysis**.  
The goal is to:

1. Retrieve reproducible raw data
2. Preserve it in original form
3. Store it in the `data/raw/` directory
4. Document metadata and station details

This notebook establishes the foundation for the entire Weather Agreement Lab.

Downstream notebooks will:
- Clean and align this data
- Measure agreement between agencies
- Build a machine learning-based fusion model

In [1]:
# If running inside a fresh environment, uncomment:
# !pip install requests pandas python-dotenv

import os
import requests
import pandas as pd
from datetime import datetime

In [2]:
import sys
import os
import platform
import subprocess
from datetime import datetime
import pkg_resources

print("===== SESSION DIAGNOSTICS =====\n")

print("Timestamp:", datetime.now())
print("\n--- Python ---")
print("Python version:", sys.version)
print("Python executable:", sys.executable)

print("\n--- Virtual Environment ---")
print("VIRTUAL_ENV:", os.environ.get("VIRTUAL_ENV"))

print("\n--- Working Directory ---")
print("Current working directory:", os.getcwd())

print("\n--- Platform ---")
print("System:", platform.system())
print("Release:", platform.release())
print("Machine:", platform.machine())
print("Processor:", platform.processor())

print("\n--- Installed Key Packages ---")
for pkg in ["pandas", "requests", "python-dotenv"]:
    try:
        version = pkg_resources.get_distribution(pkg).version
        print(f"{pkg}: {version}")
    except Exception:
        print(f"{pkg}: NOT INSTALLED")

print("\n--- Git Status ---")
try:
    result = subprocess.run(
        ["git", "rev-parse", "--abbrev-ref", "HEAD"],
        capture_output=True,
        text=True
    )
    print("Git branch:", result.stdout.strip())
except Exception:
    print("Git not available")

print("\n==============================")

===== SESSION DIAGNOSTICS =====

Timestamp: 2026-02-27 11:52:58.162880

--- Python ---
Python version: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Python executable: c:\Users\evertj\git\handson-ml3\Weather_Agreement_Lab\.venv\Scripts\python.exe

--- Virtual Environment ---
VIRTUAL_ENV: C:\Users\evertj\git\handson-ml3\Weather_Agreement_Lab\.venv

--- Working Directory ---
Current working directory: c:\Users\evertj\git\handson-ml3\Weather_Agreement_Lab

--- Platform ---
System: Windows
Release: 10
Machine: AMD64
Processor: Intel64 Family 6 Model 151 Stepping 2, GenuineIntel

--- Installed Key Packages ---
pandas: 3.0.1
requests: 2.32.5
python-dotenv: 1.2.1

--- Git Status ---
Git branch: main



In [3]:
assert ".venv" in sys.executable, "WARNING: You are not using the project virtual environment!"


In [4]:
from dotenv import load_dotenv

load_dotenv()

NOAA_TOKEN = os.getenv("NOAA_TOKEN")

if NOAA_TOKEN is None:
    raise ValueError("NOAA_TOKEN not found in .env file.")

headers = {"token": NOAA_TOKEN}

print("NOAA token loaded successfully.")

NOAA token loaded successfully.


In [5]:
test_url = "https://www.ncei.noaa.gov/cdo-web/api/v2/datasets"

response = requests.get(test_url, headers=headers)

print("Status Code:", response.status_code)

Status Code: 200


In [6]:
base_url = "https://www.ncei.noaa.gov/cdo-web/api/v2/stations"

params = {
    "datasetid": "ISD",           # Integrated Surface Database (hourly)
    "locationid": "FIPS:40",      # Oklahoma
    "startdate": "2012-01-01",
    "enddate": "2012-12-31",
    "limit": 1000
}

response = requests.get(base_url, headers=headers, params=params)

print("Status:", response.status_code)

stations = response.json()
stations_df = pd.DataFrame(stations["results"])

stations_df.head()

Status: 200


KeyError: 'results'

In [7]:
print("Status:", response.status_code)
print("Keys returned:", stations.keys())
print("Number of stations returned:", len(stations.get("results", [])))


Status: 200
Keys returned: dict_keys([])
Number of stations returned: 0


In [8]:
params = {
    "datasetid": "GHCND",
    "locationid": "FIPS:40",
    "limit": 1000
}

response = requests.get(base_url, headers=headers, params=params)

stations = response.json()
stations_df = pd.DataFrame(stations["results"])

stations_df.head()

,elevation,mindate,maxdate,latitude,name,datacoverage,id,elevationUnit,longitude
0,281.9,2008-04-06,2026-02-19,35.692100,"BUNCH 0.8 N, OK US",0.9197,GHCND:US1OKAD0002,METERS,-94.769400
1,345.0,2011-01-12,2020-06-12,35.990160,"WESTVILLE 0.2 ENE, OK US",0.9977,GHCND:US1OKAD0003,METERS,-94.571557
2,337.1,2013-04-26,2023-05-13,35.952917,"WESTVILLE 3.0 SSW, OK US",0.1897,GHCND:US1OKAD0004,METERS,-94.602300
3,338.9,2019-08-11,2026-02-15,36.077200,"WATTS 7.2 WSW, OK US",0.2033,GHCND:US1OKAD0006,METERS,-94.694440
4,346.9,2020-09-26,2026-02-14,36.751500,"JET 6.0 NNE, OK US",0.0940,GHCND:US1OKAL0002,METERS,-98.155000
